# 01 — Google Cluster 2019: ограниченная локальная выгрузка

Запускайте после `sh lab bootstrap` в локальном Jupyter. При Run All по умолчанию выполняется только вывод SQL, без сети и запросов. Синтетика в ноутбуках 02 и 03 работает без Google Cloud.

Сначала настройте проект и ADC через `sh lab configure` и `sh lab auth`; см. README и docs/DATA_ACCESS.md. Затем включите CONNECT_GOOGLE для metadata/dry run. EXECUTE_QUERIES включает отдельные запросы с расходованием квоты Sandbox с ограничением бюджета.


In [3]:
from pathlib import Path
from IPython.display import display
from vmforecast import __version__
from vmforecast.settings import Settings

SETTINGS = Settings.load()
ROOT = SETTINGS.root
LAB_DATA_DIR = SETTINGS.data
LAB_RUNS_DIR = SETTINGS.runs
print('vmforecast-lab', __version__)


vmforecast-lab 0.1.0


In [ ]:
# All dependencies and local paths are prepared by the launcher.


In [5]:
from vmforecast.google2019 import ExtractionConfig, BoundedExtractor, build_queries

CONNECT_GOOGLE = True
EXECUTE_QUERIES = False
PROJECT_ID = SETTINGS.project

CFG = ExtractionConfig(
    source_project=SETTINGS.source_project,
    cell="a",
    hours=3,
    prefix_hours=1,
    n_services=10,
    max_query_bytes=64 * 1024**3,
    total_budget_bytes=192 * 1024**3,
    max_result_rows=200_000,
)

DATA_DIR = LAB_DATA_DIR / "google2019_combined_a_3h_10"

for name, sql in build_queries(CFG).items():
    print(name, sql)


usage 
    WITH samples AS (
        SELECT start_time, end_time, collection_id, instance_index, machine_id,
               average_usage.cpus AS cpu, average_usage.memory AS mem, sample_rate
        FROM `google.com:google-cluster-data.clusterdata_2019_a.instance_usage`
        WHERE collection_type = 0
          AND (alloc_collection_id IS NULL OR alloc_collection_id = 0)
          AND machine_id > 0
          AND start_time < @t1 AND end_time > @t0 AND end_time > start_time
    ), covered AS (
        SELECT *,
               SUM(CASE WHEN start_time < @prefix_end
                             AND cpu IS NOT NULL AND mem IS NOT NULL
                        THEN LEAST(end_time, @prefix_end) - GREATEST(start_time, @t0)
                        ELSE 0 END)
               OVER (PARTITION BY collection_id, instance_index) AS prefix_coverage
        FROM samples
    ), ranked AS (
        SELECT *, DENSE_RANK() OVER (
            ORDER BY FARM_FINGERPRINT(CONCAT(CAST(collection_id AS STRING

## Авторизация

`sh lab auth` создаёт Application Default Credentials штатной утилитой Google Cloud CLI. Ноутбук использует их автоматически. Личные токены и ключи не нужно вставлять в ячейки или `.env`. При истёкшем/отозванном доступе повторите `sh lab auth` в терминале.

CONNECT_GOOGLE=False отключает сетевые действия. EXECUTE_QUERIES=False разрешает metadata и dry run, но не выполнение SQL-запросов с расходованием квоты Sandbox.


In [7]:
extractor = None
if CONNECT_GOOGLE:
    if not PROJECT_ID:
        raise ValueError('Run sh lab configure --project YOUR_PROJECT_ID, then sh lab auth')
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    extractor = BoundedExtractor(client, CFG, DATA_DIR)
    meta = extractor.metadata()
    print({name: {'rows': m['num_rows'], 'bytes': m['num_bytes']} for name, m in meta.items()})
else:
    print('Google Cloud connection disabled.')


/home/dev17/.local/state/vmforecast/d17926801ed9/venv/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


{'collection_events': {'rows': 20807441, 'bytes': 4315840386}, 'instance_events': {'rows': 1717317922, 'bytes': 428516062271}, 'machine_events': {'rows': 46219, 'bytes': 6093564}, 'machine_attributes': {'rows': 1702926, 'bytes': 133695739}, 'instance_usage': {'rows': 7575500668, 'bytes': 2259920505352}}


In [8]:
plan = extractor.plan()

GiB = 1024**3

for key in ("usage", "events_scan_upper_estimate"):
    item = plan[key]
    estimated = item["estimated_bytes"]
    print(
        key,
        "GiB:", None if estimated is None else estimated / GiB,
        "причины отказа:", item["reasons"],
    )

total = plan["total_scan_upper_estimate_bytes"]
print("Суммарная оценка, GiB:",
      None if total is None else total / GiB)
print("Помещается в заданный бюджет:",
      plan["fits_configured_budget_conservatively"])

usage GiB: 564.418782889843 причины отказа: ['per_query_limit', 'remaining_budget']
events_scan_upper_estimate GiB: 83.45754248648882 причины отказа: ['per_query_limit']
Суммарная оценка, GiB: 647.8763253763318
Помещается в заданный бюджет: False


## Проверить объединённую выгрузку до запуска

Оставьте `EXECUTE_QUERIES=False`. План показывает объединённый usage и оценку events без фильтра по ещё неизвестной когорте, даже при превышении бюджета. Это scan, а не размер скачивания. Реальные events проверяются повторно после получения ID.

Используйте новый каталог с `combined` в имени. Старый ledger сохраняйте для учёта расхода; новый каталог не обнуляет квоту Google. Не повышайте бюджеты автоматически. После выгрузки укажите этот же каталог `prepared` в notebook 02; notebook 03 по умолчанию остаётся синтетическим.


In [ ]:
if extractor is not None:
    if EXECUTE_QUERIES:
        exported = extractor.extract(execute=True)
    else:
        plan = extractor.plan()
        display(plan)
        print('Dry run only. No data queries or ledger reservations were made.')
else:
    print('Выгрузка отключена. Для синтетики откройте notebook 02.')


## Подготовка и проверка
Строгая версия требует полного покрытия каждой задачи и отсутствия перезапусков. Ошибка оставляет quality.json с причиной. Можно сократить интервал/размер когорты, явно выбрать другие ключи или отдельно реализовать lifecycle replay.

**EXPLORATORY_COMPLETE_CASE=True** — опциональный демонстрационный отбор по полноте будущих данных. Он явно маркируется как survivorship bias и сохраняет список отброшенных задач. Не использовать такой отбор для общих выводов без отдельного анализа смещения.

Ёмкости ниже синтетические, в той же нормированной шкале, что trace. Они рассчитываются только по warm-up prefix; это не восстановление реального парка Google.


In [ ]:
import numpy as np
import pandas as pd
from vmforecast.data import prepare_usage, workload_from_tidy, save_workload
EXPLORATORY_COMPLETE_CASE = False
WARMUP = 12
if (DATA_DIR/'usage.parquet').exists() and (DATA_DIR/'events.parquet').exists():
    raw = pd.read_parquet(DATA_DIR/'usage.parquet')
    events = pd.read_parquet(DATA_DIR/'events.parquet')
    tidy, qa = prepare_usage(raw, events, CFG.t0, CFG.t1,
        report_path=DATA_DIR/'quality.json', complete_case=EXPLORATORY_COMPLETE_CASE)
    print(qa)
    tidy.to_parquet(DATA_DIR/'tidy.parquet', index=False)
    n = tidy.service_id.nunique()
    fleet_size = max(2, (n+3)//4)
    prefix = tidy[tidy.tick < WARMUP]
    max_per_service = prefix.groupby('service_id')[['cpu','mem']].max()
    # Conservative starting capacity; choose denser fleets explicitly in later experiments.
    unit_cap = np.maximum(max_per_service.max().to_numpy()*1.1,
                           max_per_service.sum().to_numpy()/fleet_size*1.5)
    caps = np.tile(np.maximum(unit_cap, 1e-8), (fleet_size,1))
    workload = workload_from_tidy(tidy, caps, warmup=WARMUP, origin_us=CFG.t0,
        provenance={'source':'Google Cluster 2019', 'cell':CFG.cell,
                    'cohort_selection':qa['cohort_selection'],
                    't0':CFG.t0,'t1':CFG.t1,'demand_proxy':'average_usage'})
    save_workload(workload, DATA_DIR/'prepared')
    print('Prepared shape:',workload.demand.shape)
else:
    print('Нет выгрузки; подготовка пропущена.')
